# M7.3 · Supervised Fine-Tuning (SFT) — production recipe, single GPU

Runs the **same container-based recipe as the production SFT pipeline**
(`gsi-training/7.run_sft/4.run_sft`), scaled to **one** A100 / H100 / H200.

**Same as production:**
- Engine: **NeMo AutoModel** in **`nvcr.io/nvidia/nemo-automodel:26.04`**.
- Launch: `torchrun finetune.py --config <recipe>.yaml` (identical 4-line wrapper
  as CPT — the CPT/SFT difference lives entirely in the recipe).
- Data: **`ChatDataset`** reads chat JSONL directly, with **answer-only loss
  masking** (`start_of_turn_token`) and **sequence packing** (`packed_sequence`).

**Scaled for 1 GPU (the deltas):** small dense Nemotron, `dp_size: 1`, no
expert-parallel, FSDP2 + activation checkpointing, BF16, tiny `packed_sequence`.

**Chain:** if M7.2 produced a CPT checkpoint we fine-tune **that** (domain-adapted
model); otherwise we start from the base. Reads `data/sft_corpus.jsonl`; writes a
consolidated checkpoint to `work/sft_checkpoints/`, consumed by M7.4 (DPO).

## 1. Prerequisites — GPU, NGC login, HF token, pull the container

All training runs **inside** the container, so the host only needs Docker + a GPU
plus an NGC API key (pull) and an HF token (gated Nemotron repo).

In [1]:
import os, subprocess, getpass, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from docker_storage import (
    ensure_docker_storage,
    workshop_work_dir,
    docker_workspace_volumes,
    docker_passwd_workaround_env,
    glob_work_paths,
)
from notebook_env import bootstrap_notebook_env, ensure

ensure_docker_storage()
bootstrap_notebook_env()
ensure("torch", ["torch>=2.5.0,<2.7.0"], quiet=True)

import torch
assert torch.cuda.is_available(), "SFT needs a CUDA GPU (1x A100 / H100 / H200)."
_p = torch.cuda.get_device_properties(0)
print(f"GPU: {_p.name} ({_p.total_memory / 2**30:.1f} GiB)")

if not os.environ.get("NGC_API_KEY"):
    os.environ["NGC_API_KEY"] = getpass.getpass("NGC API key: ").strip()
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HuggingFace token (gated Nemotron repo): ").strip()
assert os.environ.get("NGC_API_KEY"), "NGC_API_KEY required to pull the container."

NB_DIR = Path.cwd().resolve()
WORK_DIR = workshop_work_dir("M7-model_training")
CONTAINER = "nvcr.io/nvidia/nemo-automodel:26.04"
BASE_MODEL = "nvidia/Nemotron-Mini-4B-Instruct"   # instruct model -> tokenizer ships a chat template

print("Docker login -> nvcr.io ...")
subprocess.run(["docker", "login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"],
               input=os.environ["NGC_API_KEY"].encode(), check=True)
print("Pulling", CONTAINER, "...")
subprocess.check_call(["docker", "pull", CONTAINER])

DOCKER = [
    "docker", "run", "--rm", "--gpus", "device=0",
    "--shm-size=16g", "--ipc=host", "--ulimit", "memlock=-1",
    "-u", f"{os.getuid()}:{os.getgid()}", "--group-add", "0",
    "-e", "HOME=/tmp",
    *docker_passwd_workaround_env(),
    "-e", "HF_HOME=/workspace/work/hf_cache",
    "-e", f"HF_TOKEN={os.environ['HF_TOKEN']}",
    "-e", "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True",
    *docker_workspace_volumes(NB_DIR, WORK_DIR),
]
print("workspace work dir (checkpoints):", WORK_DIR)

# Chain on the CPT checkpoint if M7.2 produced one, else start from the base.
_cpt = glob_work_paths(WORK_DIR, NB_DIR, "cpt_checkpoints/**/model/consolidated")
if _cpt:
    ckpt = _cpt[-1]
    if not ckpt.is_relative_to(WORK_DIR):
        # Legacy checkpoints on repo disk — extra read-only mount for the container.
        DOCKER.extend([
            "-v", f"{NB_DIR}/work/cpt_checkpoints:/workspace/work/cpt_checkpoints:ro",
        ])
        root = NB_DIR / "work"
    else:
        root = WORK_DIR  # already mounted via docker_workspace_volumes()
    MODEL_PATH = "/workspace/work/" + str(ckpt.relative_to(root))
    print("chaining SFT from CPT checkpoint:", MODEL_PATH)
else:
    MODEL_PATH = BASE_MODEL
    print("no CPT checkpoint found; SFT starts from base:", MODEL_PATH)

2026-06-16 08:44:08,380 INFO === ensure_docker_storage (storage=/data, log: /data/logs/docker_storage.log) ===
2026-06-16 08:44:08,381 INFO disk /: 132.5G used / 983.1G (13.5%)
2026-06-16 08:44:08,381 INFO disk /data: 132.5G used / 983.1G (810.3G free)
2026-06-16 08:44:08,382 INFO env TMPDIR=/data/cache/tmp
2026-06-16 08:44:08,382 INFO env DOCKER_TMPDIR=/data/cache/tmp
2026-06-16 08:44:08,383 INFO env PIP_CACHE_DIR=/data/cache/pip
2026-06-16 08:44:08,383 INFO env UV_CACHE_DIR=/data/cache/uv
2026-06-16 08:44:08,384 INFO env HF_HOME=/data/cache/hf
2026-06-16 08:44:08,384 INFO env XDG_CACHE_HOME=/data/cache/xdg
2026-06-16 08:44:08,385 INFO env LOCAL_NIM_CACHE=/data/cache/nim
2026-06-16 08:44:08,386 INFO $ docker info --format {{.DockerRootDir}}
2026-06-16 08:44:08,437 INFO docker data-root (config): /data/docker
2026-06-16 08:44:08,438 INFO docker data-root (live):   /data/docker
2026-06-16 08:44:08,439 INFO $ docker info
2026-06-16 08:44:08,491 INFO $ docker info --format {{.DockerRootDi

docker storage ok: /data/docker (810.3G free on /data)
notebook env: /home/shadeform/workshop-materials-gsi/repo-content_v2/workshop-Materials/M7-model_training/.venv (python /home/shadeform/workshop-materials-gsi/repo-content_v2/workshop-Materials/M7-model_training/.venv/bin/python, storage /data)
ok: torch


/home/shadeform/workshop-materials-gsi/repo-content_v2/workshop-Materials/M7-model_training/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


GPU: NVIDIA A100-SXM4-80GB (79.2 GiB)


NGC API key:  ········
HuggingFace token (gated Nemotron repo):  ········


Docker login -> nvcr.io ...
Login Succeeded
Pulling nvcr.io/nvidia/nemo-automodel:26.04 ...
26.04: Pulling from nvidia/nemo-automodel
Digest: sha256:7213eab8055a2029ce1ef9022384a780f9095b9577f32edca4c48d303907421f
Status: Image is up to date for nvcr.io/nvidia/nemo-automodel:26.04
nvcr.io/nvidia/nemo-automodel:26.04
workspace work dir (checkpoints): /data/workshop/M7-model_training/work
chaining SFT from CPT checkpoint: /workspace/work/cpt_checkpoints/epoch_0_step_39/model/consolidated


## 2. Write the single-GPU SFT recipe

Generated here so `/workspace/...` paths resolve in the container. Mirrors the
production `recipe_a100sxm-8.yaml` with the 1-GPU deltas.

**Parameter significance:**
- `model.pretrained_model_name_or_path` — the **CPT checkpoint** (chained) or the
  base; we **omit** `use_mamba_kernels` (dense Nemotron rejects it). We rely on the **instruct tokenizer's built-in
  chat template** (production injected one because its CPT *base* tokenizer had
  none; if you start from a base model with no template, add a `chat_template:`
  block as production does).
- `dataset: ChatDataset` + `start_of_turn_token: '<|im_start|>'` — **answer-only
  loss**: everything before the last turn marker is masked, so loss falls only on
  the assistant response, not the prompt.
- `packed_sequence.packed_sequence_size: 0` — **packing disabled** for this tiny
  corpus. With only ~20 short chats, packing collapses everything into a single
  2048-token pack, which can't fill a multi-sample global batch → the dataloader
  yields 0 full batches → `epoch_len` (and therefore `lr_decay_steps`) becomes 0
  and the LR scheduler asserts. Leaving samples unpacked keeps ~20 distinct items.
  In production (large corpus) you enable packing (e.g. 2048/8350) to avoid
  padding waste.
- `step_scheduler` — `global_batch_size: 4`, `local_batch_size: 1`, `num_epochs: 4`,
  capped at `max_steps: 20`. Optimizer steps ≈ `min(num_epochs × ⌈items/grad_acc⌉,
  max_steps)` = `min(4 × ⌈20/4⌉, 20) = 20`, comfortably above the warmup.
- `distributed` — `dp_size: 1`, no `ep_size`, FSDP2 + activation checkpointing.
- `optimizer` — AdamW `lr 1e-5`, `betas (0.9, 0.999)` (the SFT convention, vs CPT's
  `0.95` beta2); cosine LR with lr_warmup_steps: 2 (must stay below total optimizer steps).

In [2]:
SFT_RECIPE = f'''# Single-GPU SFT recipe (workshop). Mirrors gsi-training/7.run_sft/4.run_sft
# scaled to 1 GPU + a small dense Nemotron. Driven by recipes/finetune.py.
model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: {MODEL_PATH}
  trust_remote_code: true
  torch_dtype: bfloat16
  # NOTE: no `use_mamba_kernels` -- that flag is only for the Mamba-hybrid
  # Nemotron-H (30B production model); a dense NemotronForCausalLM rejects it.
  # backend:
  #   _target_: nemo_automodel.components.models.common.BackendConfig
  #   linear: te
  #   rms_norm: torch_fp32
  #   enable_hf_state_dict_adapter: true
  #   enable_fsdp_optimizations: true

fp8:
  enabled: false

dataset:
  _target_: nemo_automodel.components.datasets.llm.chat_dataset.ChatDataset
  path_or_dataset_id: /workspace/data/sft_corpus.jsonl
  split: train
  start_of_turn_token: "<|im_start|>"   # answer-only loss masking boundary

validation_dataset:
  _target_: nemo_automodel.components.datasets.llm.chat_dataset.ChatDataset
  path_or_dataset_id: /workspace/data/sft_corpus.jsonl
  split: train
  start_of_turn_token: "<|im_start|>"

step_scheduler:
  global_batch_size: 4
  local_batch_size: 1
  ckpt_every_steps: 10
  val_every_steps: 10
  num_epochs: 1
  max_steps: 20

dist_env:
  backend: nccl
  timeout_minutes: 60

rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 1111
  ranked: true

checkpoint:
  enabled: true
  checkpoint_dir: /workspace/work/sft_checkpoints/
  model_save_format: safetensors
  save_consolidated: true

distributed:
  strategy: fsdp2
  dp_size: 1
  dp_replicate_size: 1
  tp_size: 1
  cp_size: 1
  pp_size: 1
  activation_checkpointing: true
  sequence_parallel: false

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

packed_sequence:
  packed_sequence_size: 0   # disabled for this tiny corpus: with only ~20 short
                            # chats, packing collapses them into 1 pack, leaving
                            # too few optimizer steps (epoch_len -> 0). Keeping
                            # samples unpacked yields ~20 distinct training items.
                            # In production (large corpus) set this to 2048/8350.

dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater
  shuffle: true
  num_workers: 0
  pin_memory: true

validation_dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater
  num_workers: 0
  pin_memory: true

optimizer:
  _target_: torch.optim.AdamW
  betas: [0.9, 0.999]
  eps: 1.0e-8
  lr: 1.0e-5
  weight_decay: 0.1

lr_scheduler:
  lr_decay_style: cosine
  lr_warmup_steps: 2
  min_lr: 1.0e-6
'''
(NB_DIR / "recipes" / "sft_1gpu.yaml").write_text(SFT_RECIPE)
print("wrote recipes/sft_1gpu.yaml | model =", MODEL_PATH)


wrote recipes/sft_1gpu.yaml | model = /workspace/work/cpt_checkpoints/epoch_0_step_39/model/consolidated


## 3. Run SFT (`torchrun finetune.py` in the container)

Same launch as CPT, `--nproc-per-node=1`. Watch the loss trend down; because of
answer-only masking, the loss reflects how well the model predicts the *assistant*
turns only.

In [3]:
import sys
import sys
from docker_storage import docker_passwd_workaround_env

proc = subprocess.Popen(
    DOCKER + docker_passwd_workaround_env() + [
        "--workdir", "/workspace/recipes", CONTAINER,
        "torchrun", "--nproc-per-node=1",
        "/workspace/recipes/finetune.py",
        "--config", "/workspace/recipes/sft_1gpu.yaml",
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    sys.stdout.write(line)
rc = proc.wait()
print("\nSFT exited with code", rc)



== PyTorch ==

NVIDIA Release 26.02 (build 305635159)
PyTorch Version 2.11.0a0+eb65b36
Container image Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
Copyright (c) 2014-2024 Facebook Inc.
Copyright (c) 2011-2014 Idiap Research Institute (Ronan Collobert)
Copyright (c) 2012-2014 Deepmind Technologies    (Koray Kavukcuoglu)
Copyright (c) 2011-2012 NEC Laboratories America (Koray Kavukcuoglu)
Copyright (c) 2011-2013 NYU                      (Clement Farabet)
Copyright (c) 2006-2010 NEC Laboratories America (Ronan Collobert, Leon Bottou, Iain Melvin, Jason Weston)
Copyright (c) 2006      Idiap Research Institute (Samy Bengio)
Copyright (c) 2001-2004 Idiap Research Institute (Ronan Collobert, Samy Bengio, Johnny Mariethoz)
Copyright (c) 2015      Google Inc.
Copyright (c) 2015      Yangqing Jia
Copyright (c) 2013-2016 The Caffe contributors
All rights reserved.

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

GOVERN

## 4. Inspect the checkpoint

The consolidated checkpoint under `work/sft_checkpoints/` is the SFT output that
M7.4 (DPO) aligns.

In [4]:
ckpt_root = WORK_DIR / "sft_checkpoints"
print("checkpoints:")
for d in sorted(ckpt_root.glob("*")):
    print("  ", d.name)
consolidated = sorted(ckpt_root.glob("**/model/consolidated"))
print("\nconsolidated checkpoint:", consolidated[-1] if consolidated else "(none yet)")
print("Next: M7.4 rl.ipynb aligns this with DPO.")


checkpoints:
   LATEST
   LOWEST_VAL
   epoch_0_step_4
   epoch_1_step_9
   epoch_2_step_14
   epoch_3_step_19
   training.jsonl
   validation.jsonl

consolidated checkpoint: (none yet)
Next: M7.4 rl.ipynb aligns this with DPO.
